# 🏎️ Fabric Racing Game - Play!

HTML5 multiplayer racing game with real-time telemetry.

**Controls:**
- ⬆️ Arrow Up: Accelerate
- ⬇️ Arrow Down: Brake
- ⬅️ Arrow Left: Steer Left
- ➡️ Arrow Right: Steer Right

In [ ]:
# Configuration - Update these values!
EVENTSTREAM_ENDPOINT = "<YOUR_CUSTOM_ENDPOINT_URL>"
PLAYER_ID = "Player1"  # Change for each player (Player1, Player2, Player3, Player4)
CAR_COLOR = "red"      # red, blue, green, yellow

In [ ]:
from IPython.display import HTML, display
import uuid

session_id = str(uuid.uuid4())
print(f"🏎️ Game Session: {session_id[:8]}...")
print(f"👤 Player: {PLAYER_ID}")
print(f"🚗 Car Color: {CAR_COLOR}")

In [ ]:
game_html = f'''
<!DOCTYPE html>
<html>
<head>
    <style>
        #gameCanvas {{
            border: 3px solid #333;
            border-radius: 10px;
            background: #2d5a27;
            display: block;
            margin: 10px auto;
        }}
        #gameInfo {{
            font-family: 'Courier New', monospace;
            text-align: center;
            padding: 10px;
            background: #1a1a2e;
            color: #00ff00;
            border-radius: 5px;
            margin: 10px auto;
            max-width: 800px;
        }}
        #controls {{
            text-align: center;
            padding: 10px;
            font-family: Arial, sans-serif;
        }}
        .stat {{
            display: inline-block;
            margin: 0 20px;
            font-size: 18px;
        }}
        .stat-label {{ color: #888; }}
        .stat-value {{ color: #fff; font-weight: bold; }}
    </style>
</head>
<body>
    <div id="gameInfo">
        <span class="stat"><span class="stat-label">LAP:</span> <span id="lap" class="stat-value">0</span>/3</span>
        <span class="stat"><span class="stat-label">SPEED:</span> <span id="speed" class="stat-value">0</span> km/h</span>
        <span class="stat"><span class="stat-label">POSITION:</span> <span id="position" class="stat-value">-</span></span>
        <span class="stat"><span class="stat-label">TIME:</span> <span id="time" class="stat-value">0:00</span></span>
    </div>
    <canvas id="gameCanvas" width="800" height="500"></canvas>
    <div id="controls">
        <p>⬆️ Accelerate | ⬇️ Brake | ⬅️➡️ Steer | <strong>SPACE</strong> Start Race</p>
    </div>

    <script>
        const canvas = document.getElementById("gameCanvas");
        const ctx = canvas.getContext("2d");
        const ENDPOINT = "{EVENTSTREAM_ENDPOINT}";
        const PLAYER_ID = "{PLAYER_ID}";
        const CAR_COLOR = "{CAR_COLOR}";
        const SESSION_ID = "{session_id}";
        
        // Track parameters (oval)
        const trackCenterX = 400;
        const trackCenterY = 250;
        const trackRadiusX = 300;
        const trackRadiusY = 180;
        const trackWidth = 80;
        
        // Car state
        let car = {{
            angle: 0,
            speed: 0,
            lap: 0,
            lastSection: -1,
            x: trackCenterX + trackRadiusX,
            y: trackCenterY
        }};
        
        let keys = {{}};
        let gameStarted = false;
        let raceStartTime = 0;
        let eventCount = 0;
        
        // Input handling
        document.addEventListener("keydown", (e) => {{
            keys[e.key] = true;
            if (e.key === " " && !gameStarted) {{
                startRace();
            }}
            e.preventDefault();
        }});
        document.addEventListener("keyup", (e) => {{
            keys[e.key] = false;
        }});
        
        function startRace() {{
            gameStarted = true;
            raceStartTime = Date.now();
            car.lap = 0;
            car.lastSection = -1;
            sendEvent("RaceStart");
        }}
        
        function sendEvent(eventType) {{
            if (!ENDPOINT.startsWith("http")) return;
            
            const event = {{
                EventId: crypto.randomUUID(),
                Timestamp: new Date().toISOString(),
                PlayerId: PLAYER_ID,
                EventType: eventType,
                CarId: ["red", "blue", "green", "yellow"].indexOf(CAR_COLOR) + 1,
                PositionX: Math.round(car.x * 100) / 100,
                PositionY: Math.round(car.y * 100) / 100,
                Speed: Math.round(car.speed * 10) / 10,
                LapNumber: car.lap,
                TrackSection: getTrackSection(),
                GameSessionId: SESSION_ID
            }};
            
            fetch(ENDPOINT, {{
                method: "POST",
                headers: {{ "Content-Type": "application/json" }},
                body: JSON.stringify(event)
            }}).catch(e => console.log("Send error:", e));
            
            eventCount++;
        }}
        
        function getTrackSection() {{
            const normalizedAngle = ((car.angle % (2 * Math.PI)) + 2 * Math.PI) % (2 * Math.PI);
            const section = Math.floor(normalizedAngle / (Math.PI / 2));
            return ["StartFinish", "Turn1", "Backstraight", "Turn2"][section];
        }}
        
        function update() {{
            if (!gameStarted) return;
            
            // Acceleration/braking
            if (keys["ArrowUp"]) car.speed = Math.min(200, car.speed + 2);
            else if (keys["ArrowDown"]) car.speed = Math.max(0, car.speed - 3);
            else car.speed = Math.max(0, car.speed - 0.5);
            
            // Steering
            const turnRate = 0.03 * (car.speed / 100);
            if (keys["ArrowLeft"]) car.angle -= turnRate;
            if (keys["ArrowRight"]) car.angle += turnRate;
            
            // Update position on track
            car.x = trackCenterX + trackRadiusX * Math.cos(car.angle);
            car.y = trackCenterY + trackRadiusY * Math.sin(car.angle);
            
            // Lap detection
            const currentSection = Math.floor((car.angle / (2 * Math.PI)) * 4) % 4;
            if (car.lastSection === 3 && currentSection === 0) {{
                car.lap++;
                sendEvent("LapComplete");
                if (car.lap >= 3) {{
                    sendEvent("RaceEnd");
                    gameStarted = false;
                    alert("🏁 Race Complete! Check KQL for results.");
                }}
            }}
            car.lastSection = currentSection;
            
            // Send position updates
            if (eventCount % 5 === 0) sendEvent("Position");
        }}
        
        function draw() {{
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            
            // Draw grass
            ctx.fillStyle = "#2d5a27";
            ctx.fillRect(0, 0, canvas.width, canvas.height);
            
            // Draw track
            ctx.strokeStyle = "#444";
            ctx.lineWidth = trackWidth;
            ctx.beginPath();
            ctx.ellipse(trackCenterX, trackCenterY, trackRadiusX, trackRadiusY, 0, 0, 2 * Math.PI);
            ctx.stroke();
            
            // Draw track lines
            ctx.strokeStyle = "#fff";
            ctx.lineWidth = 2;
            ctx.setLineDash([20, 15]);
            ctx.beginPath();
            ctx.ellipse(trackCenterX, trackCenterY, trackRadiusX, trackRadiusY, 0, 0, 2 * Math.PI);
            ctx.stroke();
            ctx.setLineDash([]);
            
            // Draw start/finish line
            ctx.strokeStyle = "#fff";
            ctx.lineWidth = 5;
            ctx.beginPath();
            ctx.moveTo(trackCenterX + trackRadiusX - trackWidth/2, trackCenterY - 5);
            ctx.lineTo(trackCenterX + trackRadiusX + trackWidth/2, trackCenterY - 5);
            ctx.stroke();
            
            // Draw car
            ctx.save();
            ctx.translate(car.x, car.y);
            ctx.rotate(car.angle + Math.PI/2);
            
            // Car body
            ctx.fillStyle = CAR_COLOR;
            ctx.fillRect(-10, -15, 20, 30);
            ctx.fillStyle = "#333";
            ctx.fillRect(-8, -10, 16, 8); // cockpit
            
            ctx.restore();
            
            // Update HUD
            document.getElementById("lap").textContent = car.lap;
            document.getElementById("speed").textContent = Math.round(car.speed);
            document.getElementById("position").textContent = getTrackSection();
            
            if (gameStarted) {{
                const elapsed = Math.floor((Date.now() - raceStartTime) / 1000);
                const mins = Math.floor(elapsed / 60);
                const secs = elapsed % 60;
                document.getElementById("time").textContent = `${{mins}}:${{secs.toString().padStart(2, "0")}}`;
            }}
            
            // "Press SPACE" message
            if (!gameStarted) {{
                ctx.fillStyle = "rgba(0,0,0,0.7)";
                ctx.fillRect(250, 200, 300, 80);
                ctx.fillStyle = "#fff";
                ctx.font = "24px Arial";
                ctx.textAlign = "center";
                ctx.fillText("Press SPACE to Start", 400, 250);
            }}
        }}
        
        function gameLoop() {{
            update();
            draw();
            requestAnimationFrame(gameLoop);
        }}
        
        gameLoop();
    </script>
</body>
</html>
'''

display(HTML(game_html))

## 📊 View Your Race Data

After racing, run these KQL queries to see your telemetry:

```kql
// Your lap times
GameEvents
| where PlayerId == "Player1"  // Change to your player ID
| where EventType == "LapComplete"
| project LapNumber, Timestamp

// Speed distribution by track section
GameEvents
| where EventType == "Position"
| summarize AvgSpeed = avg(Speed) by TrackSection, PlayerId
| render columnchart

// Position heatmap
GameEvents
| where EventType == "Position"
| project PositionX, PositionY, Speed
| render scatterchart
```